# Transformer — Attention Is All You Need (2017)

_Papers · Transformers & LLMs_

**Drop recurrence and convolution; build a sequence model out of attention, positional encoding, and residual+LayerNorm blocks.**

---

This notebook is a practice scaffold for the **Transformer — Attention Is All You Need (2017)** lesson. We build it up one piece at a time: the attention and position primitives, the multi-head and encoder blocks, then a tiny end-to-end model with a clean ablation. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Hand-check the two core primitives

Before assembling anything, we verify the two ideas the whole architecture rests on. First, **scaled dot-product attention** split across two heads: each head looks at a different 2-column slice of the input, runs softmax over the dot-product scores, and the per-head outputs are concatenated. Second, the **sinusoidal positional encoding** — a fixed table of sines and cosines at geometrically spaced frequencies that gives every position a unique fingerprint. Computing both by hand on a tiny `d_model=4` example makes the numbers concrete before we wrap them in modules.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

# --- Worked example: 2-head split (d_model=4, h=2, d_k=2), identity projections. ---
X = torch.tensor([[2., 0., 1., 1.],
                  [0., 2., 1., -1.]])          # 2 tokens, d_model=4

def sdpa(Q, K, V):                              # scaled dot-product attention (your paper-attention primitive)
    dk = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / math.sqrt(dk)
    weights = F.softmax(scores, dim=-1)
    return weights @ V

head1 = sdpa(X[:, 0:2], X[:, 0:2], X[:, 0:2])   # head 1 = columns 0,1
head2 = sdpa(X[:, 2:4], X[:, 2:4], X[:, 2:4])   # head 2 = columns 2,3
concat = torch.cat([head1, head2], dim=-1)
print("head1 token0 =", [round(v, 4) for v in head1[0].tolist()])      # [1.8884, 0.1116]
print("head2 token0 =", [round(v, 4) for v in head2[0].tolist()])      # [1.0, 0.6089]
print("concat token0 =", [round(v, 4) for v in concat[0].tolist()])    # [1.8884, 0.1116, 1.0, 0.6089]

# --- Worked example: sinusoidal positional encoding (d_model=4). ---
def positional_encoding(seq_len, d_model):
    pos = torch.arange(seq_len).unsqueeze(1).float()            # (seq_len, 1)
    i2 = torch.arange(0, d_model, 2).float()                    # 0, 2, 4, ...  (the "2i")
    denom = torch.pow(10000.0, i2 / d_model)                    # 10000^(2i/d_model)
    pe = torch.zeros(seq_len, d_model)
    pe[:, 0::2] = torch.sin(pos / denom)                        # even cols: sin
    pe[:, 1::2] = torch.cos(pos / denom)                        # odd cols:  cos
    return pe

pe = positional_encoding(2, 4)
print("PE(pos=0) =", [round(v, 4) for v in pe[0].tolist()])     # [0.0, 1.0, 0.0, 1.0]
print("PE(pos=1) =", [round(v, 4) for v in pe[1].tolist()])     # [0.8415, 0.5403, 0.01, 0.9999]

### Step 2 — Build multi-head attention and the encoder block

Now we wrap the primitive in a module. **Multi-head attention** projects the input to queries/keys/values, reshapes them into `h` parallel heads of width `d_k = d_model / h`, runs scaled dot-product attention per head, then concatenates and mixes with an output projection `W^O`. The **encoder block** wraps a sublayer in the paper's `LayerNorm(x + Sublayer(x))` pattern: a residual connection plus normalization, applied once around attention and once around the position-wise feed-forward network.

In [ ]:
# --- Multi-head attention, built by hand (reshape -> per-head SDPA -> concat -> W^O). ---
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, h):
        super().__init__()
        assert d_model % h == 0
        self.h = h
        self.d_k = d_model // h
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def _split(self, x):                                 # (B, S, d_model) -> (B, h, S, d_k)
        B, S, _ = x.shape
        return x.view(B, S, self.h, self.d_k).transpose(1, 2)

    def forward(self, x):
        Q, K, V = self.W_q(x), self.W_k(x), self.W_v(x)
        Q = self._split(Q)
        K = self._split(K)
        V = self._split(V)
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)   # \S 3.2.1 scaling = sqrt(d_k)
        attn = F.softmax(scores, dim=-1)
        out = attn @ V                                           # (B, h, S, d_k)
        B, _, S, _ = out.shape
        out = out.transpose(1, 2).contiguous().view(B, S, self.h * self.d_k)  # concat heads
        return self.W_o(out)                                     # \S 3.2.2 multi-head + W^O


# --- Encoder block: LayerNorm(x + Sublayer(x)) for attention and feed-forward (\S 3.1, 3.3). ---
class EncoderBlock(nn.Module):
    def __init__(self, d_model, h, d_ff):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, h)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        x = self.norm1(x + self.attn(x))      # residual + LayerNorm around attention
        x = self.norm2(x + self.ff(x))        # residual + LayerNorm around feed-forward
        return x

### Step 3 — Assemble the model and the reverse task

The `TinyTransformer` stacks an embedding, the positional-encoding table, a few encoder blocks, and an output projection. The `use_pe` flag toggles the single line that adds positions to the embeddings — that is the knob for the ablation in Step 4. The task is **reverse**: the target is the input read backwards. We deliberately avoid plain copy, because copying needs no position information (each output equals the same-position input), so it would not expose the positional-encoding ablation; reversing genuinely requires knowing where each token sits.

In [ ]:
# --- A tiny Transformer encoder for the REVERSE task. use_pe toggles the ablation. ---
class TinyTransformer(nn.Module):
    def __init__(self, vocab, d_model=32, h=4, d_ff=64, n_layers=2, max_len=12, use_pe=True):
        super().__init__()
        self.use_pe = use_pe
        self.embed = nn.Embedding(vocab, d_model)
        self.register_buffer("pe", positional_encoding(max_len, d_model))
        self.blocks = nn.ModuleList([EncoderBlock(d_model, h, d_ff) for _ in range(n_layers)])
        self.out = nn.Linear(d_model, vocab)

    def forward(self, tokens):
        x = self.embed(tokens)
        if self.use_pe:
            x = x + self.pe[:x.shape[1]]      # the single line the ablation removes
        for blk in self.blocks:
            x = blk(x)
        return self.out(x)                    # (B, S, vocab) -> predict the target token at each position


# --- Toy REVERSE task: target is the input sequence backwards. ---
# (We use reverse, not plain copy: copying needs no position -- each output = same-position input --
#  so it would NOT expose the positional-encoding ablation. Reversing requires position, so it does.)
VOCAB, SEQ, B = 10, 8, 128

def batch():
    tokens = torch.randint(1, VOCAB, (B, SEQ))    # symbols 1..VOCAB-1 (0 reserved/unused)
    targets = torch.flip(tokens, dims=[1])        # reverse: target = input read backwards
    return tokens, targets

### Step 4 — Train with and without positional encoding

Same training loop run twice, identical except for the `use_pe` flag — an honest ablation that changes exactly one thing. With positions on, the model learns to reverse and token accuracy climbs toward ~1.0. With positions off, it plateaus near chance: self-attention is a permutation-invariant weighted sum, so without a position signal the model literally cannot tell which slot to fetch when reversing.

In [ ]:
def train(use_pe, steps=800, lr=3e-3):
    torch.manual_seed(0)
    net = TinyTransformer(VOCAB, use_pe=use_pe)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    lf = nn.CrossEntropyLoss()
    for s in range(steps):
        x, y = batch()
        logits = net(x)
        loss = lf(logits.reshape(-1, VOCAB), y.reshape(-1))
        opt.zero_grad()
        loss.backward()
        opt.step()
        if s % 200 == 0 or s == steps - 1:
            acc = (logits.argmax(-1) == y).float().mean().item()
            print(f"  step {s:4d}  loss {loss.item():.4f}  token-acc {acc:.3f}")
    return acc

print("\nWITH positional encoding (use_pe=True):")
acc_pe = train(use_pe=True)
print("WITHOUT positional encoding  (ABLATION, use_pe=False):")
acc_no = train(use_pe=False)
print(f"\nfinal token accuracy  PE-on: {acc_pe:.3f}   PE-off: {acc_no:.3f}")
# PE-on climbs toward ~1.0 (it learns to reverse); PE-off plateaus near chance (~0.32 in our run)
# because self-attention is permutation-invariant and cannot tell which position to fetch.
# (Exact numbers vary by hardware/seed; this is our small run, not the paper's reported result.)

## Visualize the data & results

_On a reverse task, does the tiny Transformer learn with positional encoding, and does removing it (ablation) destroy the ability to recover order?_

### Step 1 — Rebuild the model compactly

This panel is self-contained so it can run on its own. We re-declare the positional encoding and a compact version of the same multi-head attention, encoder block, and tiny Transformer — identical logic to the reference implementation, just renamed tersely. Re-seeding keeps the run reproducible.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

def positional_encoding(seq_len, d_model):
    pos = torch.arange(seq_len).unsqueeze(1).float()
    i2 = torch.arange(0, d_model, 2).float()
    denom = torch.pow(10000.0, i2 / d_model)
    pe = torch.zeros(seq_len, d_model)
    pe[:, 0::2] = torch.sin(pos / denom)
    pe[:, 1::2] = torch.cos(pos / denom)
    return pe

class MHA(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.h, self.dk = h, d // h
        self.Wq, self.Wk, self.Wv, self.Wo = (nn.Linear(d, d) for _ in range(4))
    def split(self, x):
        B, S, _ = x.shape
        return x.view(B, S, self.h, self.dk).transpose(1, 2)
    def forward(self, x):
        Q = self.split(self.Wq(x))
        K = self.split(self.Wk(x))
        V = self.split(self.Wv(x))
        a = F.softmax(Q @ K.transpose(-2, -1) / math.sqrt(self.dk), dim=-1) @ V
        B, _, S, _ = a.shape
        return self.Wo(a.transpose(1, 2).contiguous().view(B, S, self.h * self.dk))

class Block(nn.Module):
    def __init__(self, d, h, ff):
        super().__init__()
        self.a = MHA(d, h)
        self.f = nn.Sequential(nn.Linear(d, ff), nn.ReLU(), nn.Linear(ff, d))
        self.n1, self.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
    def forward(self, x):
        x = self.n1(x + self.a(x))
        return self.n2(x + self.f(x))

class T(nn.Module):
    def __init__(self, V, d=32, h=4, ff=64, L=2, mx=12, use_pe=True):
        super().__init__()
        self.use_pe = use_pe
        self.e = nn.Embedding(V, d)
        self.register_buffer("pe", positional_encoding(mx, d))
        self.b = nn.ModuleList([Block(d, h, ff) for _ in range(L)])
        self.o = nn.Linear(d, V)
    def forward(self, t):
        x = self.e(t)
        if self.use_pe:
            x = x + self.pe[:x.shape[1]]
        for blk in self.b:
            x = blk(x)
        return self.o(x)

### Step 2 — Train both variants and record the curve

We run the same reverse task with positional encoding on and off, recording token accuracy at every step so we can compare the two learning curves. With positions, accuracy jumps to 1.0 within roughly a hundred steps; without them it stays flat near chance (~0.32), the model unable to recover order at all.

In [ ]:
V, S, B = 10, 8, 128

def batch():
    t = torch.randint(1, V, (B, S))
    return t, torch.flip(t, dims=[1])             # reverse task

def run(use_pe, steps=800):
    torch.manual_seed(0)
    net = T(V, use_pe=use_pe)
    opt = torch.optim.Adam(net.parameters(), lr=3e-3)
    lf = nn.CrossEntropyLoss()
    accs = []
    for s in range(steps):
        x, y = batch()
        lg = net(x)
        loss = lf(lg.reshape(-1, V), y.reshape(-1))
        opt.zero_grad()
        loss.backward()
        opt.step()
        accs.append((lg.argmax(-1) == y).float().mean().item())
    return accs

on = run(True)
off = run(False)
idx = list(range(0, 800, 50)) + [799]
print("PE on :", [[i, round(on[i], 3)] for i in idx])
print("PE off:", [[i, round(off[i], 3)] for i in idx])
# PE on -> jumps to 1.0 by ~step 100 (learns to reverse). PE off -> flat near chance ~0.32 (order lost).

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. Your tiny Transformer reverses a sequence correctly with positional encoding
            ON (it reaches ~100% token accuracy). Remove the single line that adds the $PE$ table to the
            embeddings and retrain. What happens to the accuracy, and what does that demonstrate about
            self-attention?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Delete only the position signal: change x = embed(tokens) + PE to x = embed(tokens); keep depth, width, heads, optimizer, data, and seed identical. — _An honest ablation changes exactly one thing &mdash; the positional encoding &mdash; so any difference is attributable to it._
- Retrain and watch token accuracy: with $PE$ ON it climbs toward ~100%; with $PE$ OFF it plateaus near chance (~0.32 in our run) and never reverses. — _Reversing needs the output at position $i$ to fetch the input at position $N{-}1{-}i$ &mdash; impossible without a position signal, since self-attention is a permutation-invariant weighted sum._
- Conclude that positional encoding, not extra capacity, is what lets the attention-only model respect order. — _Both runs share architecture and parameter count; only the $+PE$ differs, isolating it as the cause._

**Answer:** With positional encoding removed, accuracy collapses to near chance &mdash; the model can still
                 learn which symbols appear but cannot place them in reversed order, because
                 self-attention is permutation-invariant (a weighted sum does not depend on position). Since
                 the two runs are
                 identical except for the "$+PE$", this isolates positional encoding as the source of order
                 information. The CODEVIZ panel shows exactly this contrast.

</details>

**Problem 2.** In the worked example ($d_{\text{model}}=4$, $h=2$), head 1 put weight $[0.944, 0.056]$ on the two
            tokens while head 2 put $[0.804, 0.196]$. Why is it useful that the two heads attend
            differently, and what would $h=1$ (a single full-width head) cost you?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note that each head ran on a different 2-dim slice of the input, so they "saw" different content and produced different attention weights. — _That is multi-head's purpose (\S 3.2.2): jointly attend to different representation subspaces at once._
- Imagine collapsing to one head over all 4 dims: you get a single attention distribution per query, one lens on the sequence. — _A single averaged attention can only express one relationship at a time; the paper says this "inhibits" attending to several things._
- Confirm the cost is the same: $h$ heads of width $d_{\text{model}}/h$ total the same compute as one head of width $d_{\text{model}}$. — _Splitting buys expressivity (multiple lenses) for free._

**Answer:** Different heads give the model several attention patterns in one pass &mdash; e.g. one head
                 tracking a nearby token, another a distant one. A single head ($h=1$) forces one averaged
                 distribution per query, which the paper notes inhibits attending to multiple things at once.
                 Because each head is $1/h$ as wide, the total compute is unchanged, so multi-head buys
                 expressivity essentially for free.

</details>

**Problem 3.** Your positional-encoding check gave $PE_{(0)} = [0,1,0,1]$ and
            $PE_{(1)} \approx [0.8415, 0.5403, 0.0100, 1.0]$ for $d_{\text{model}}=4$. Why is column 2
            ($0.0100$) so much smaller than column 0 ($0.8415$) at the same position, and what does that
            spread of frequencies buy the model?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Read the formula: column $0$ uses $\sin(pos/10000^{0})=\sin(pos)$; column $2$ uses $\sin(pos/10000^{1/2})=\sin(pos/100)$. — _The exponent $2i/d_{\text{model}}$ grows with the column, so the angular frequency shrinks &mdash; later columns are slow waves._
- At $pos=1$: $\sin(1)\approx0.8415$ but $\sin(0.01)\approx0.01$. — _The slow wave has barely moved from position $0$, so its value is near $0$._
- Conclude the table mixes fast and slow waves, so each position gets a unique multi-scale fingerprint and nearby positions stay similar on slow channels. — _Multi-scale frequencies let the model read both fine and coarse position, and (via angle-addition) shifts become linear &mdash; aiding relative-position attention._

**Answer:** The exponent $2i/d_{\text{model}}$ makes higher columns lower-frequency: column 0 is
                 $\sin(pos)$ (fast), column 2 is $\sin(pos/100)$ (slow), so at $pos=1$ the slow wave is still
                 near $0$ ($\sin(0.01)\approx0.01$). Spanning many frequencies gives every position a unique,
                 smoothly varying fingerprint at multiple scales, and the angle-addition identities make a
                 shift by $k$ a fixed linear map &mdash; which is why the paper expected it to help the model
                 attend by relative position.

</details>